In [ ]:
# 0. Setup

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# --- Make src.* importable from any working directory ---
_root = Path.cwd()
while not (_root / 'src').is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.paths import get_base_path

BASE_PATH = get_base_path()

FILES = {
    'full_history':       'full_history.parquet',
    'eda_ready':          'eda_ready.parquet',
    'tracks_exportify':   'tracks_exportify.parquet',
    'orphans_unified':    'orphans_unified.parquet',
    'playlist_links':     'playlist_links.parquet',
    'playlists_metadata': 'playlists_metadata.parquet',
    'library_tracks':     'library_tracks.parquet',
    'owner_ids':          'owner_ids.parquet',
    'track_dates':        'track_dates.parquet',
}

print(f'🔍 Quality check · {len(FILES)} files')

🔍 Quality check · 9 files


In [ ]:
# 1. Load files

dfs = {}
for name, fname in FILES.items():
    path = BASE_PATH / fname
    if path.exists():
        dfs[name] = pd.read_parquet(path)
        print(f'✅ {name}: {len(dfs[name])} rows, {len(dfs[name].columns)} cols')
    else:
        dfs[name] = None
        print(f'❌ {name}: missing')

✅ full_history: 79752 rows, 48 cols
✅ eda_ready: 79752 rows, 49 cols
✅ tracks_exportify: 24953 rows, 29 cols
✅ orphans_unified: 18742 rows, 23 cols
✅ playlist_links: 33094 rows, 8 cols
✅ playlists_metadata: 13417 rows, 13 cols
✅ library_tracks: 2567 rows, 7 cols
✅ owner_ids: 3 rows, 2 cols
✅ track_dates: 10315 rows, 3 cols


In [ ]:
# 2. Time consistency

print('\n🕒 Time consistency\n')

# --- 2.1 ts_utc presence and ts − ts_utc diff ---
for name in ['full_history', 'eda_ready']:
    df = dfs.get(name)
    if df is None:
        print(f'⚠️ {name}: not loaded')
        continue
    if 'ts_utc' not in df.columns:
        print(f'❌ {name}: ts_utc missing')
        continue
    diff = (df['ts'] - df['ts_utc']).dropna().abs().max()
    status = '✅' if diff == pd.Timedelta(0) else '⚠️'
    print(f'{status} {name}: ts - ts_utc max diff = {diff}')

# --- 2.2 track_dates datetime range ---
td = dfs.get('track_dates')
if td is not None:
    for col in ['library_added_at', 'first_playlist_added_at']:
        if col not in td.columns:
            print(f'⚠️ track_dates.{col}: missing column')
            continue
        if not pd.api.types.is_datetime64_any_dtype(td[col]):
            print(f'❌ track_dates.{col}: not datetime')
            continue
        years = td[col].dt.year.dropna()
        ok = not years.empty and years.between(2020, 2026).all()
        status = '✅' if ok else '⚠️'
        print(f'{status} track_dates.{col}: years in 2020–2026')
else:
    print('⚠️ track_dates: not loaded')


🕒 Time consistency

✅ full_history: ts - ts_utc max diff = 0 days 00:00:00
✅ eda_ready: ts - ts_utc max diff = 0 days 00:00:00
✅ track_dates.library_added_at: years in 2020–2026
✅ track_dates.first_playlist_added_at: years in 2020–2026


In [ ]:
# 3. Clean data

print('\n🧹 Clean data\n')

# --- 3.1 Empty URIs ---
uri_checks = [
    ('tracks_exportify', 'spotify_track_uri'),
    ('orphans_unified',  'spotify_track_uri'),
    ('full_history',     'spotify_track_uri'),
    ('eda_ready',        'spotify_track_uri'),
    ('library_tracks',   'spotify_track_uri'),
    ('playlist_links',   'Track URI'),
]
for name, col in uri_checks:
    df = dfs.get(name)
    if df is None:
        print(f'⚠️ {name}: not loaded')
        continue
    if col not in df.columns:
        print(f'⚠️ {name}.{col}: missing column')
        continue
    empty = df[col].isna().sum() + (df[col] == '').sum()
    status = '✅' if empty == 0 else '❌'
    print(f'{status} {name}.{col}: {empty} empty')

# --- 3.2 Missing values ---
for name in ['tracks_exportify', 'full_history', 'eda_ready']:
    df = dfs.get(name)
    if df is None:
        continue
    for col in ['genres', 'genres_normalized']:
        if col in df.columns:
            nan = df[col].isna().sum()
            status = '✅' if nan == 0 else '❌'
            print(f'{status} {name}.{col}: {nan} NaN')

eda = dfs.get('eda_ready')
if eda is not None and 'played_frac' in eda.columns:
    nan = eda['played_frac'].isna().sum()
    inf = np.isinf(eda['played_frac']).sum()
    status = '✅' if nan == 0 and inf == 0 else '❌'
    print(f'{status} eda_ready.played_frac: NaN={nan}, inf={inf}')

for name in ['full_history', 'eda_ready']:
    df = dfs.get(name)
    if df is None:
        continue
    for col in ['duration_ms', 'ms_played']:
        if col in df.columns:
            nan = df[col].isna().sum()
            status = '✅' if nan == 0 else '❌'
            print(f'{status} {name}.{col}: {nan} NaN')

# --- 3.3 Duration == 0 ---
for name in ['full_history', 'eda_ready']:
    df = dfs.get(name)
    if df is None or 'duration_ms' not in df.columns:
        continue
    zero = (df['duration_ms'] == 0).sum()
    status = '✅' if zero == 0 else '❌'
    print(f'{status} {name}.duration_ms == 0: {zero}')


🧹 Clean data

✅ tracks_exportify.spotify_track_uri: 0 empty
✅ orphans_unified.spotify_track_uri: 0 empty
✅ full_history.spotify_track_uri: 0 empty
✅ eda_ready.spotify_track_uri: 0 empty
✅ library_tracks.spotify_track_uri: 0 empty
✅ playlist_links.Track URI: 0 empty
✅ tracks_exportify.genres: 0 NaN
✅ tracks_exportify.genres_normalized: 0 NaN
✅ full_history.genres: 0 NaN
✅ full_history.genres_normalized: 0 NaN
✅ eda_ready.genres: 0 NaN
✅ eda_ready.genres_normalized: 0 NaN
✅ eda_ready.played_frac: NaN=0, inf=0
✅ full_history.duration_ms: 0 NaN
✅ full_history.ms_played: 0 NaN
✅ eda_ready.duration_ms: 0 NaN
✅ eda_ready.ms_played: 0 NaN
✅ full_history.duration_ms == 0: 0
✅ eda_ready.duration_ms == 0: 0


In [ ]:
# 4. Keys & links

print('\n🔑 Keys & links\n')

# --- 4.1 Duplicates ---
te = dfs.get('tracks_exportify')
if te is not None and 'spotify_track_uri' in te.columns:
    dup = te.duplicated(subset=['spotify_track_uri']).sum()
    print(f"{'✅' if dup == 0 else '❌'} tracks_exportify: dup URIs = {dup}")

pl = dfs.get('playlist_links')
if pl is not None:
    req = ['name_norm', 'Track URI', 'Added By', 'Added At',
           'playlist_type', 'has_external_adders']
    miss = [c for c in req if c not in pl.columns]
    if miss:
        print(f'❌ playlist_links: missing {miss} '
              f'(check 00_1 and 00_2)')
    else:
        tech = pl.duplicated(subset=req[:4]).sum()  # dupes on base cols
        print(f"{'✅' if tech == 0 else '❌'} playlist_links: tech dupes = {tech}")

pm = dfs.get('playlists_metadata')

if pm is not None:
    full = pm.duplicated().sum()
    print(f'✅ playlists_metadata: full dupes = {full} (allowed)')

lt = dfs.get('library_tracks')
if lt is not None:
    dup = lt.duplicated(subset=['spotify_track_uri']).sum()
    print(f'✅ library_tracks: URI repeats = {dup} (allowed)')

oi = dfs.get('owner_ids')
if oi is not None and 'owner_id' in oi.columns:
    dup = oi.duplicated(subset=['owner_id']).sum()
    print(f"{'✅' if dup == 0 else '❌'} owner_ids: dup owner_id = {dup}")

# --- 4.2 URI integrity ---
if te is not None and 'spotify_track_uri' in te.columns:
    te_uris = set(te['spotify_track_uri'].dropna())
    for name, col in [('playlist_links', 'Track URI'), ('library_tracks', 'spotify_track_uri')]:
        df = dfs.get(name)
        if df is None or col not in df.columns:
            continue
        uris = set(df[col].dropna())
        missing = uris - te_uris
        status = '✅' if not missing else '⚠️'
        print(f'{status} {name}: {len(missing)} URIs missing in tracks_exportify')


🔑 Keys & links

✅ tracks_exportify: dup URIs = 0
✅ playlist_links: tech dupes = 0
✅ playlists_metadata: full dupes = 79 (allowed)
✅ library_tracks: URI repeats = 209 (allowed)
✅ owner_ids: dup owner_id = 0
✅ playlist_links: 0 URIs missing in tracks_exportify
✅ library_tracks: 0 URIs missing in tracks_exportify


In [ ]:
# 5. Ranges

print('\n📊 Ranges\n')

eda = dfs.get('eda_ready')
if eda is None:
    print('❌ eda_ready not loaded')
else:
    checks = {
        'valence':          (0, 1),
        'energy':           (0, 1),
        'danceability':     (0, 1),
        'acousticness':     (0, 1),
        'instrumentalness': (0, 1),
        'liveness':         (0, 1),
        'speechiness':      (0, 1),
        'popularity':       (0, 100),
        'tempo':            (10, 300),
        'ms_played':        (0, np.inf),
        'played_frac':      (0, np.inf),
    }

    for col, (low, high) in checks.items():
        if col not in eda.columns:
            print(f'⚠️ {col}: missing column')
            continue
        mn, mx = eda[col].min(), eda[col].max()
        ok = (mn >= low) and (mx <= high) and not np.isinf(mx)
        status = '✅' if ok else '❌'
        print(f'{status} {col}: [{mn:.3f}, {mx:.3f}] (expected [{low}, {high}])')

    for col in ['tempo', 'popularity']:
        if col in eda.columns:
            zeros = (eda[col] == 0).sum()
            nan = eda[col].isna().sum()
            status = '✅' if zeros == 0 else '❌'
            print(f'{status} {col}: zeros={zeros}, NaN={nan} (zeros must be 0)')


📊 Ranges

✅ valence: [0.000, 0.983] (expected [0, 1])
✅ energy: [0.000, 1.000] (expected [0, 1])
✅ danceability: [0.000, 0.985] (expected [0, 1])
✅ acousticness: [0.000, 0.996] (expected [0, 1])
✅ instrumentalness: [0.000, 1.000] (expected [0, 1])
✅ liveness: [0.000, 0.987] (expected [0, 1])
✅ speechiness: [0.000, 0.960] (expected [0, 1])
✅ popularity: [1.000, 95.000] (expected [0, 100])
✅ tempo: [33.426, 241.426] (expected [10, 300])
✅ ms_played: [0.000, 3669268.000] (expected [0, inf])
✅ played_frac: [0.000, 4.033] (expected [0, inf])
✅ tempo: zeros=0, NaN=58 (zeros must be 0)
✅ popularity: zeros=0, NaN=17206 (zeros must be 0)


In [ ]:
# 6. Flags consistency

print('\n🔗 Flags consistency\n')

eda = dfs.get('eda_ready')
fh = dfs.get('full_history')

if eda is None or fh is None:
    print('❌ eda_ready or full_history not loaded')
else:
    # --- 6.1 Short plays must be skipped ---
    short = eda[eda['ms_played'] < 10000]
    not_skip_short = (short['is_skip'] == False).sum()
    status = '✅' if not_skip_short == 0 else '❌'
    print(f'{status} short plays (<10s): {len(short)}, not skipped: {not_skip_short}')

    # --- 6.2 Late skips are allowed ---
    long_skip = ((eda['ms_played'] >= 10000) & (eda['is_skip'] == True)).sum()
    print(f'✅ late skips (>=10s): {long_skip} (allowed)')

    # --- 6.3 No analytical columns in eda_ready ---
    analytical_cols = [
        'valid_play', 'is_base_obsession', 'is_strong_obsession',
        'first_valid_seen', 'first_listen_valid', 'first_seen_valid'
    ]
    present = [c for c in analytical_cols if c in eda.columns]
    status = '✅' if not present else '❌'
    print(f'{status} analytical columns in eda_ready: {present}')

    # --- 6.4 first_listen rules ---
    not_orphan_mask = (fh['source'] != 'exportify_orphans') | (fh['orphan_category'] == 'not_orphan')
    not_orphan_with_first = fh[not_orphan_mask & fh['first_listen'].notna()]

    orphans_with_metrics = fh[
        (fh['source'] == 'exportify_orphans') &
        (fh['orphan_category'].isin(['skipped', 'significant', 'insignificant']))
    ]
    orphans_without_first = orphans_with_metrics[orphans_with_metrics['first_listen'].isna()]

    s1 = '✅' if len(not_orphan_with_first) == 0 else '❌'
    s2 = '✅' if len(orphans_without_first) == 0 else '❌'
    print(f'{s1} first_listen on not_orphans: {len(not_orphan_with_first)} (expected 0)')
    print(f'{s2} first_listen missing on orphans: {len(orphans_without_first)} (expected 0)')


🔗 Flags consistency

✅ short plays (<10s): 20427, not skipped: 0
✅ late skips (>=10s): 7580 (allowed)
✅ analytical columns in eda_ready: []
✅ first_listen on not_orphans: 0 (expected 0)
✅ first_listen missing on orphans: 0 (expected 0)


In [ ]:
# 7. Forbidden columns

print('\n🚫 Forbidden columns\n')

podcast_cols = [
    'episode_name', 'episode_show_name', 'spotify_episode_uri',
    'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title'
]

for name in ['full_history', 'eda_ready']:
    df = dfs.get(name)
    if df is None:
        continue
    found = [c for c in podcast_cols if c in df.columns]
    status = '✅' if not found else '❌'
    print(f'{status} {name}: {found if found else "none"}')


🚫 Forbidden columns

✅ full_history: none
✅ eda_ready: none


In [ ]:
# 8. UNTRANSLATED artifacts

print('\n🔎 UNTRANSLATED artifacts\n')

for name in ['tracks_exportify', 'full_history', 'eda_ready']:
    df = dfs.get(name)
    if df is None or 'genres_normalized' not in df.columns:
        print(f'⚠️ {name}: missing genres_normalized')
        continue
    count = df['genres_normalized'].str.startswith('UNTRANSLATED:', na=False).sum()
    status = '✅' if count == 0 else '❌'
    print(f'{status} {name}: {count} rows with UNTRANSLATED')


🔎 UNTRANSLATED artifacts

✅ tracks_exportify: 0 rows with UNTRANSLATED
✅ full_history: 0 rows with UNTRANSLATED
✅ eda_ready: 0 rows with UNTRANSLATED


In [ ]:
# 9. Summary

print('\n=== Summary ===')
print(f'Files checked: {sum(1 for v in dfs.values() if v is not None)}/{len(FILES)}')
print('All checks completed. If everything above is ✅ or acceptable ⚠️,')
print('data is ready for the EDA stage (01–04).')


=== Summary ===
Files checked: 9/9
All checks completed. If everything above is ✅ or acceptable ⚠️,
data is ready for the EDA stage (01–04).


## Содержание `quality_check.ipynb`

**Цель:** проверить целостность данных после `00_1` и `00_2` перед EDA.

**Принципы:** пути через `src.paths`, вывод — только счётчики и статусы ✅/❌, без имён аккаунтов и путей.

---

### Что проверяется

**0–1. Setup и загрузка.**
Импорты, `BASE_PATH`, словарь `FILES`, загрузка 9 паркетов в `dfs`. Вывод — строки и колонки.

**2. Time consistency.**
- `ts_utc` есть в `full_history` и `eda_ready`
- `ts − ts_utc = 0` при `UTC_OFFSET_HOURS=0`
- даты в `track_dates` в диапазоне 2020–2026

**3. Clean data.**
- пустые URI во всех ключевых файлах
- `NaN` в жанрах, `played_frac` (и `inf`), `duration_ms`, `ms_played`
- `duration_ms == 0` — не должно быть

**4. Keys & links.**
- дубликаты по правилам (URI в `tracks_exportify`, техдубли в `playlist_links`, `owner_id` в `owner_ids`)
- допустимые повторы (`library_tracks`, `playlists_metadata`)
- целостность URI между файлами

**5. Ranges.**
Аудио-характеристики в [0,1], `popularity` [0,100], `tempo` [10,300]. Нули в `tempo`/`popularity` — 0 (заменены на NaN).

**6. Flags consistency.**
- все `<10s` прослушивания имеют `is_skip=True`
- в `eda_ready` нет аналитических колонок (`valid_play`, `is_base_obsession` и др.)
- `first_listen` заполнен только у настоящих сирот

**7. Forbidden columns.**
Нет колонок подкастов/аудиокниг.

**8. UNTRANSLATED artifacts.**
Нет строк с `UNTRANSLATED:` в `genres_normalized`.

**9. Summary.**
Сколько файлов загружено, готовность к EDA.

---


## Содержание `00_3_quality_check.ipynb`

**Цель ноутбука:**
Проверить целостность данных после `00_1` и `00_2` перед EDA и дашбордом. Найти проблемы: пропущенные значения, неверные диапазоны, потерянные связи, утечки приватных колонок.

**Общие принципы:**
- Пути через `src.paths`, без хардкода.
- В выводе — только статусы ✅ / ⚠️ / ❌ и счётчики. Никаких имён аккаунтов, никаких таблиц, никаких путей.
- Ноутбук **не меняет** данные — только читает и проверяет.
- Если что-то не так — видно, **где именно** и **сколько**.

---

### Ячейка 0. Setup

**Что делает:**
- Импортирует библиотеки, подключает `src/`.
- Загружает `BASE_PATH`.
- Определяет словарь `FILES` — 9 паркетов для проверки.
- Печатает, сколько файлов будет проверено.

**Выход:** `BASE_PATH`, `FILES`.

---

### Ячейка 1. Загрузка файлов

**Вход:** все 9 паркетов из `FILES`.
**Выход:** словарь `dfs` — имя → DataFrame или None.

**Логика:**
- Для каждого файла из `FILES` проверяет существование.
- Если есть — читает в parquet, печатает строки × колонки.
- Если нет — `None`, печатает `❌ missing`.

**Печатает:** статус по каждому файлу.

---

### Ячейка 2. Time consistency

**Что проверяет:**
- `ts_utc` присутствует в `full_history` и `eda_ready`.
- `ts − ts_utc = 0` при `UTC_OFFSET_HOURS = 0`.
- Даты в `track_dates` в диапазоне 2020–2026.
- Колонки `library_added_at` и `first_playlist_added_at` — datetime.

**Печатает:** статус по каждому пункту.

---

### Ячейка 3. Clean data

**Что проверяет:**
- **Пустые URI** — во всех файлах, где есть колонка с URI (`tracks_exportify`, `orphans_unified`, `full_history`, `eda_ready`, `library_tracks`, `playlist_links`).
- **NaN в жанрах** — в `tracks_exportify`, `full_history`, `eda_ready`.
- **NaN / inf в `played_frac`** — в `eda_ready`.
- **NaN в `duration_ms` и `ms_played`** — в `full_history`, `eda_ready`.
- **`duration_ms == 0`** — не должно быть.

**Печатает:** статус и число проблемных строк по каждому пункту.

---

### Ячейка 4. Keys & links

**Что проверяет:**
- **Дубликаты по правилам:**
  - `tracks_exportify`: дубли по `spotify_track_uri` — должно быть 0.
  - `playlist_links`: техдубли по `(name_norm, Track URI, Added By, Added At)` — должно быть 0.
  - `owner_ids`: дубли по `owner_id` — должно быть 0.
  - `library_tracks`, `playlists_metadata`: повторы разрешены.
- **Целостность URI:** все URI из `playlist_links` и `library_tracks` должны присутствовать в `tracks_exportify`. Если что-то отсутствует — предупреждение.

**Печатает:** статус и число по каждому пункту.

---

### Ячейка 5. Ranges

**Что проверяет:**
- Аудио-фичи в `eda_ready` в диапазонах:
  - `valence`, `energy`, `danceability`, `acousticness`, `instrumentalness`, `liveness`, `speechiness` — [0, 1]
  - `popularity` — [0, 100]
  - `tempo` — [10, 300]
  - `ms_played`, `played_frac` — [0, ∞)
- Нули в `tempo` и `popularity` — не должно быть (они были заменены на NaN в `00_1`).

**Печатает:** min, max и статус по каждому признаку.

---

### Ячейка 6. Flags consistency

**Что проверяет:**
- **Короткие прослушивания** (`ms_played < 10s`) — все должны иметь `is_skip=True`.
- **Длинные скипы** (`ms_played >= 10s` и `is_skip=True`) — разрешены (пользователь мог скипнуть позже).
- **Отсутствие аналитических колонок** в `eda_ready`: `valid_play`, `is_base_obsession`, `is_strong_obsession`, `first_valid_seen`, `first_listen_valid`, `first_seen_valid`.
- **`first_listen` только у настоящих сирот:**
  - у не-сирот `first_listen` не заполнен — должно быть 0;
  - у сирот с категорией (`skipped` / `significant` / `insignificant`) `first_listen` заполнен — пропусков быть не должно.

**Печатает:** статус по каждому пункту.

---

### Ячейка 7. Forbidden columns

**Что проверяет:**
- В `full_history` и `eda_ready` нет колонок подкастов / аудиокниг:
  - `episode_name`, `episode_show_name`, `spotify_episode_uri`
  - `audiobook_title`, `audiobook_uri`, `audiobook_chapter_uri`, `audiobook_chapter_title`

**Печатает:** какие колонки найдены (должно быть пусто).

---

### Ячейка 8. UNTRANSLATED artifacts

**Что проверяет:**
- В `tracks_exportify`, `full_history`, `eda_ready` в колонке `genres_normalized` нет строк, начинающихся с `UNTRANSLATED:`. Если есть — значит `genre_mapping.json` не покрыл какие-то жанры.

**Печатает:** число строк с `UNTRANSLATED:`.

---

### Ячейка 9. Summary

**Что делает:**
- Считает, сколько файлов загружено (из 9).
- Печатает итог: если всё ✅ — данные готовы к EDA (ноутбуки 01–04).

**Выход:** сводка.

---

### Итог

Ноутбук **ничего не создаёт и не меняет**. Только читает и проверяет. Пользователь запускает его после `00_1` и `00_2` и получает отчёт о состоянии всех паркетов.

Если всё ✅ — можно идти в EDA. Если что-то ❌ — понятно, на каком уровне проблема: данные, связи, диапазоны или флаги.

---

## Что решает ноутбук

`00_3` решает задачу **контроля качества перед анализом**. Пайплайн длинный: `00_0` → `00_1` → `00_2` — на каждом шаге могут появиться проблемы. Exportify вернул CSV с пустой колонкой `valence`? JOIN потерял часть событий? Категоризация проставила `first_listen` не тем, кому надо? `genre_mapping` не покрыл какие-то жанры?

Без этой проверки ошибка уходит дальше — в EDA, в дашборд, в портрет. И через месяц никто не понимает, почему Sound-кластеры странные или почему Retention в компасе в два раза ниже, чем ожидалось. Дёшево поймать проблему **сейчас**, дорого — через месяц.

Вторая задача — **защита от тихих ошибок**. Тихая ошибка хуже явной: пайплайн не падает, работает, но результат неверный. `00_3` делает такие ошибки **громкими**. Например, если `playlist_links` потерял 347 треков между запусками — ноутбук покажет расхождение, а не молча продолжит.

## Как решает

Девять независимых блоков проверок, каждый — свой аспект:

- **Загрузка** — все ли файлы на месте.
- **Time consistency** — не съехали ли часовые пояса и границы дат.
- **Clean data** — нет ли NaN, пустых URI, нулевых длительностей.
- **Keys & links** — не потерялись ли связи между таблицами.
- **Ranges** — аудио-фичи в допустимых границах или вылезли за [0,1].
- **Flags consistency** — правила `is_skip`, категоризация сирот.
- **Forbidden columns** — не просочились ли подкасты / приватные поля.
- **UNTRANSLATED** — нормализованы ли жанры.
- **Summary** — итог.

Каждый блок работает **независимо**. Если упал один — остальные всё равно проверяются. В конце — общий отчёт.

Ключевое отличие от обычного `assert`: ноутбук **не падает**. Он собирает **все** проблемы и показывает разом. Это правильнее — иначе пользователь чинит одно, запускает снова, находит второе, и так десять раз.

## Что даёт

**Отчёт о состоянии данных.** Пользователь видит:
- Все ли 9 паркетов на месте.
- Нет ли критичных ошибок (❌).
- Есть ли предупреждения (⚠️) и насколько они серьёзные.

**Разделение ошибок и нормы.** Не всё, что выглядит странно, — ошибка. Например, `library_tracks` с дублями по URI — это **правильно** (трек лайкнут в двух аккаунтах), ноутбук явно помечает это как «allowed». А `duration_ms == 0` — это **ошибка**, её быть не должно.

**Точку невозврата.** Если `00_3` показывает всё ✅ — можно спокойно идти в EDA. Если что-то ❌ — сначала чинить. Это правило.

Ноутбук **не блокирует** пайплайн. Даже с ошибками можно запустить EDA — но результат будет содержать эти ошибки. Ответственность на пользователе: он видит отчёт и решает сам.

По объёму — это самый короткий ноутбук из `00_prep`. Но по важности — один из ключевых. Он работает как **приёмка**: убедиться, что предыдущие шаги сделали то, что должны, прежде чем строить на них что-то дальше.